In [ ]:
import csv
import math
import time
from collections import Counter
from pathlib import Path
import numpy as np
from rdflib import Graph, URIRef, RDF, RDFS
from rdflib.namespace import SKOS
import difflib

try:
    import nltk
    from nltk.corpus import wordnet as wn
except ImportError:
    pass


In [ ]:

class MOSAIC:

    def __init__(self, lex_thres=0.60, sem_thres=0.72):
        self.lex_thres = lex_thres
        self.sem_thres = sem_thres
        self.model = None  
        self._wn_ready = False

    def init_wordnet(self):
        if not self._wn_ready:
            try:
                import nltk
                from nltk.corpus import wordnet as wn
                wn.synsets("test")
            except (ImportError, LookupError):
                nltk.download("wordnet", quiet=True)
                nltk.download("omw-1.4", quiet=True)
            self._wn_ready = True

    def load_ontology(self, path: Path) -> Graph:
        g = Graph()
        fmt = "turtle" if path.suffix == ".ttl" else "xml"
        try:
            g.parse(str(path), format=fmt)
            return g
        except Exception as e:
            print(f" [MOSAIC] Failed to load {path.name}: {e}")
            return None

    def extract_entities(self, graph: Graph):
        self.init_wordnet()
        from nltk.corpus import wordnet as wn

        entities = {}
        c_uri = URIRef("http://www.w3.org/2002/07/owl#Class")
        o_uri = URIRef("http://www.w3.org/2002/07/owl#ObjectProperty")
        d_uri = URIRef("http://www.w3.org/2002/07/owl#DatatypeProperty")
        i_uri = URIRef("http://www.w3.org/2002/07/owl#NamedIndividual")
        valid_types = {c_uri, o_uri, d_uri, i_uri}

        for s in graph.subjects():
            if isinstance(s, URIRef):
                etype = graph.value(s, RDF.type)
                if etype in valid_types:
                    label = (
                        graph.value(s, RDFS.label)
                        or graph.value(s, SKOS.prefLabel)
                        or s.split("#")[-1]
                    )
                    lbl_str = str(label).lower()
                    tokens = set(lbl_str.replace("-", " ").replace("_", " ").split())
                    
                    syns = set()
                    for t in tokens:
                        for syn in wn.synsets(t):
                            for lm in syn.lemmas():
                                syns.add(lm.name().lower().replace("_", " "))

                    entities[s] = {
                        "label": lbl_str,
                        "type": etype,
                        "tokens": tokens,
                        "synonyms": syns
                    }
        return entities

    def get_ngrams(self, text: str):
        grams = []
        n = len(text)
        if n >= 3:
            grams.extend(text[i : i + 3] for i in range(n - 2))
        if n >= 4:
            grams.extend(text[i : i + 4] for i in range(n - 3))
        return Counter(grams) if grams else Counter([text])

    def compute_ngram_cosine(self, s_grams, t_grams):
        shared = set(s_grams.keys()) & set(t_grams.keys())
        if not shared:
            return 0.0
            
        dot = sum(s_grams[g] * t_grams[g] for g in shared)
        s_norm = math.sqrt(sum(v**2 for v in s_grams.values()))
        t_norm = math.sqrt(sum(v**2 for v in t_grams.values()))

        return dot / (s_norm * t_norm) if s_norm and t_norm else 0.0

    def compute_isub(self, s_lbl: str, t_lbl: str):
        if not s_lbl or not t_lbl:
            return 0.0

        sm = difflib.SequenceMatcher(None, s_lbl, t_lbl)
        match = sm.find_longest_match(0, len(s_lbl), 0, len(t_lbl))
        lcs = match.size

        comm = (2 * lcs) / (len(s_lbl) + len(t_lbl))
        u_s = len(s_lbl) - lcs
        u_t = len(t_lbl) - lcs
        
        denom = (0.6 * len(s_lbl) * len(t_lbl) + 0.4 * (u_s + u_t))
        diff = (u_s * u_t) / denom if denom > 0 else 0.0

        pfx = 0.0
        for k in range(min(len(s_lbl), len(t_lbl), 4)):
            if s_lbl[k] == t_lbl[k]:
                pfx += 0.1
            else:
                break

        sim = comm - diff
        return sim + (0.1 * pfx * (1.0 - sim))

    def lexical_blocking(self, src_ents, tgt_ents):
        candidates = []

        # Token-based inverted index for quick target lookups
        inv_idx = {}
        for t_uri, t_meta in tgt_ents.items():
            for t in t_meta["tokens"]:
                inv_idx.setdefault(t, []).append((t_uri, t_meta))

        for s_uri, s_meta in src_ents.items():
            s_lbl = s_meta["label"]
            s_toks = s_meta["tokens"]
            s_grams = self.get_ngrams(s_lbl)
            matches = {}
            
            # 1. Direct token intersection
            for t in s_toks:
                if t in inv_idx:
                    for t_uri, t_meta in inv_idx[t]:
                        if s_meta["type"] == t_meta["type"]:
                            matches[t_uri] = t_meta

            # 2. Substring & Synonym fallback scans
            for t_uri, t_meta in tgt_ents.items():
                if s_meta["type"] != t_meta["type"] or t_uri in matches:
                    continue
                
                t_lbl = t_meta["label"]
                if s_lbl in t_lbl or t_lbl in s_lbl:
                    matches[t_uri] = t_meta
                elif s_meta["synonyms"] & t_meta["tokens"]:
                    matches[t_uri] = t_meta

            # 3. Score generation for survivors
            for t_uri, t_meta in matches.items():
                t_lbl = t_meta["label"]
                t_grams = self.get_ngrams(t_lbl)

                ngram_score = self.compute_ngram_cosine(s_grams, t_grams)
                isub_score = self.compute_isub(s_lbl, t_lbl)
                lev_score = difflib.SequenceMatcher(None, s_lbl, t_lbl).ratio()

                score = (0.4 * isub_score) + (0.3 * lev_score) + (0.3 * ngram_score)
                if s_lbl in t_lbl or t_lbl in s_lbl:
                    score += 0.15

                if score >= self.lex_thres:
                    candidates.append({
                        "source": s_uri,
                        "source_label": s_lbl,
                        "target": t_uri,
                        "target_label": t_lbl,
                        "score": min(1.0, score),
                        "type": s_meta["type"],
                    })

        return candidates

    def semantic_similarity(self, candidates):
        if not candidates:
            return []

        if self.model is None:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("all-MiniLM-L6-v2")

        lbls = list(set([c["source_label"] for c in candidates] + [c["target_label"] for c in candidates]))
        vecs = self.model.encode(lbls, show_progress_bar=False)
        v_dict = dict(zip(lbls, vecs))

        scored = []
        for c in candidates:
            v_src = v_dict[c["source_label"]]
            v_tgt = v_dict[c["target_label"]]

            dot = np.dot(v_src, v_tgt)
            n_a = np.linalg.norm(v_src)
            n_b = np.linalg.norm(v_tgt)
            
            sem_score = dot / (n_a * n_b) if (n_a * n_b) > 0 else 0.0
            final_score = (0.4 * c["score"]) + (0.6 * sem_score)

            if final_score >= self.sem_thres:
                c["combined_score"] = final_score
                scored.append(c)

        return scored

    def extract_final_alignments(self, scored_pairs):
        sorted_pairs = sorted(scored_pairs, key=lambda x: x["combined_score"], reverse=True)
        final_pool = []
        used_src = set()
        used_tgt = set()

        for c in sorted_pairs:
            if c["source"] not in used_src and c["target"] not in used_tgt:
                used_src.add(c["source"])
                used_tgt.add(c["target"])
                final_pool.append(c)

        return final_pool

    def align(self, src_graph: Graph, tgt_graph: Graph):
        src_ents = self.extract_entities(src_graph)
        tgt_ents = self.extract_entities(tgt_graph)

        candidates = self.lexical_blocking(src_ents, tgt_ents)
        sem_candidates = self.semantic_similarity(candidates)
        final_pool = self.extract_final_alignments(sem_candidates)

        alignments = set()
        eq_class = "http://www.w3.org/2002/07/owl#equivalentClass"
        eq_prop = "http://www.w3.org/2002/07/owl#equivalentProperty"
        same_as = "http://www.w3.org/2002/07/owl#sameAs"

        c_uri = URIRef("http://www.w3.org/2002/07/owl#Class")
        o_uri = URIRef("http://www.w3.org/2002/07/owl#ObjectProperty")
        d_uri = URIRef("http://www.w3.org/2002/07/owl#DatatypeProperty")
        i_uri = URIRef("http://www.w3.org/2002/07/owl#NamedIndividual")

        for c in final_pool:
            s_uri, t_uri, etype = c["source"], c["target"], c["type"]
            if etype == c_uri:
                alignments.add((str(s_uri), eq_class, str(t_uri)))
            elif etype in [o_uri, d_uri]:
                alignments.add((str(s_uri), eq_prop, str(t_uri)))
            elif etype == i_uri:
                alignments.add((str(s_uri), same_as, str(t_uri)))

        return alignments


class OAEITrackRunner:

    def __init__(self, matcher: MOSAIC):
        self.matcher = matcher
        self.log = []

    def load_reference_alignments(self, path: Path) -> set:
        ref_set = set()
        g = Graph()
        try:
            g.parse(str(path), format="turtle")
            valid_preds = {
                "http://www.w3.org/2002/07/owl#equivalentClass",
                "http://www.w3.org/2000/01/rdf-schema#subClassOf",
                "http://www.w3.org/2002/07/owl#equivalentProperty",
                "http://www.w3.org/2000/01/rdf-schema#subPropertyOf",
                "http://www.w3.org/2002/07/owl#sameAs",
            }
            for s, p, o in g:
                if str(p) in valid_preds:
                    nodes = sorted([str(s), str(o)])
                    ref_set.add((nodes[0], str(p), nodes[1]))
        except Exception as e:
            print(f" Could not read reference file {path.name}: {e}")
        return ref_set

    def serialize_alignments_to_ttl(self, alignments: set, path: Path):
        g = Graph()
        for src, pred, tgt in alignments:
            g.add((URIRef(src), URIRef(pred), URIRef(tgt)))
        try:
            g.serialize(destination=str(path), format="turtle")
            print(f"   [MOSAIC] Output saved to: {path.parent.name}/{path.name}")
        except Exception as e:
            print(f"   [MOSAIC] Serialization error: {e}")

    def calculate_metrics(self, sys_align, ref_align):
        if not ref_align:
            return 0.0, 0.0, 0.0

        sys_canon = set()
        for s, p, o in sys_align:
            nodes = sorted([str(s), str(o)])
            sys_canon.add((nodes[0], str(p), nodes[1]))

        tp = len(sys_canon.intersection(ref_align))
        p = tp / len(sys_canon) if sys_canon else 0.0
        r = tp / len(ref_align) if ref_align else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

        return round(p, 4), round(r, 4), round(f1, 4)

    def run_all_tracks(self, base_dir: str, csv_out: str = "mosaic_evaluation_report.csv"):
        start_global = time.time()
        base_path = Path(base_dir)
        res_dir = Path("../results")
        res_dir.mkdir(parents=True, exist_ok=True)

        if not base_path.exists():
            print(f"Error: Base directory '{base_dir}' does not exist.")
            return

        for track in base_path.iterdir():
            if not track.is_dir():
                continue

            print(f"\n" + "=" * 50)
            print(f" TRACK RUNNER: {track.name.upper()}")
            print(f"=" * 50)

            tasks = list(track.glob("*.ttl"))
            p_sum, r_sum, f_sum, t_sum = 0.0, 0.0, 0.0, 0.0
            count = 0

            for tf in tasks:
                parts = tf.stem.split("-")
                if len(parts) != 2:
                    continue

                src_p = track / "ontologies" / f"{parts[0]}.owl"
                if not src_p.exists():
                    src_p = track / "ontologies" / f"{parts[0]}.rdf"

                tgt_p = track / "ontologies" / f"{parts[1]}.owl"
                if not tgt_p.exists():
                    tgt_p = track / "ontologies" / f"{parts[1]}.rdf"

                print(f"\nMOSAIC Task: {parts[0]} ➔ {parts[1]}")

                if not src_p.exists() or not tgt_p.exists():
                    print(" Skipping task. Missing ontology file.")
                    continue

                ref_align = self.load_reference_alignments(tf)
                src_g = self.matcher.load_ontology(src_p)
                tgt_g = self.matcher.load_ontology(tgt_p)

                if src_g and tgt_g:
                    t0 = time.time()
                    alignments = self.matcher.align(src_g, tgt_g)
                    dt = round(time.time() - t0, 2)
                    
                    print(f" Step complete. MOSAIC returned {len(alignments)} matches in {dt}s.")

                    out_ttl = res_dir / f"mosaic_{track.name}_{tf.name}"
                    self.serialize_alignments_to_ttl(alignments, out_ttl)

                    p, r, f1 = self.calculate_metrics(alignments, ref_align)
                    print(f"   Metrics -> Precision: {p}, Recall: {r}, F1-Score: {f1} (Time: {dt}s)")

                    self.log.append({
                        "Track": track.name,
                        "Task": tf.stem,
                        "Precision": p,
                        "Recall": r,
                        "F1-Score": f1,
                        "Time (s)": dt,
                        "Type": "Task",
                    })

                    p_sum += p
                    r_sum += r
                    f_sum += f1
                    t_sum += dt
                    count += 1

            if count > 0:
                avg_p = round(p_sum / count, 4)
                avg_r = round(r_sum / count, 4)
                avg_f1 = round(f_sum / count, 4)
                avg_t = round(t_sum / count, 2)

                print(f"\n Track [{track.name}] AVERAGES -> P: {avg_p}, R: {avg_r}, F1: {avg_f1} | Avg Time: {avg_t}s")

                self.log.append({
                    "Track": track.name,
                    "Task": "TRACK_AVERAGE",
                    "Precision": avg_p,
                    "Recall": avg_r,
                    "F1-Score": avg_f1,
                    "Time (s)": avg_t,
                    "Type": "Average",
                })

        total_runtime = round(time.time() - start_global, 2)
        print(f"\n" + "=" * 50)
        print(f" RUN COMPLETION: Finished in {total_runtime}s.")
        print(f"=" * 50)

        self.log.append({
            "Track": "ALL_TRACKS",
            "Task": "TOTAL_PROGRAM_TIME",
            "Precision": "",
            "Recall": "",
            "F1-Score": "",
            "Time (s)": total_runtime,
            "Type": "Summary",
        })

        self.results_to_csv(csv_out)

    def results_to_csv(self, filename: str):
        fields = ["Track", "Task", "Precision", "Recall", "F1-Score", "Time (s)", "Type"]
        with open(filename, mode="w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.log)
        print(f" Compilation written to: {filename}")


if __name__ == "__main__":
    m = MOSAIC(lex_thres=0.60, sem_thres=0.72)
    runner = OAEITrackRunner(matcher=m)
    runner.run_all_tracks("../tracks", csv_out="mosaic_evaluation_report.csv")